# 📊 Week 1 — EDA Exercises

**Objective**: Practice the concepts learned in the EDA example notebook.

**Instructions**:
- Complete the exercises below by filling in the code cells marked with `# YOUR CODE HERE`
- Use the example notebook as a reference when needed
- Focus on understanding the "why" behind each step, not just the "how"

**Dataset**: Use the same sample data: `../sample_data_from_redshift/sample.parquet` or extract any other table you want using the following Redshift extraction:

```python
USER_NAME = "federico"  # ⚠️ CHANGE THIS! Example: "john_doe"

# Redshift and S3 Configuration (matches test_redshift_connection.ipynb)
# ================================================
REDSHIFT_CONFIG = {
    'cluster_id': 'redshift-cluster-dsi',
    'database': 'prod',
    'db_user': 'svc_sagemaker',
    'region': 'af-south-1',
    'iam_role': 'arn:aws:iam::733246370304:role/RedshiftIAMAuthRole',
    's3_export_prefix': f"s3://sagemaker-af-south-1-733246370304/redshift_exports/{USER_NAME}/eda_train_{date.today():%Y-%m-%d}",
    'cleanup_s3': False  # Set to True to delete temp files after loading
}

# SQL Query - Use training features with RANDOM() sampling for EDA
# ================================================
SQL_QUERY = """
SELECT *
FROM dth_churn_ml_training.training_features
WHERE RANDOM() < 0.0001;
"""

# Load data using boto3 redshift-data API pattern (same as test notebook)
# Sampling is handled by RANDOM() in the SQL query
df = load_data(
    source='redshift',
    sql=SQL_QUERY,
    redshift_kwargs=REDSHIFT_CONFIG
)
```

## Setup

Import the necessary libraries.

In [ ]:
# YOUR CODE HERE
# Import: pandas, numpy, plotly.express, seaborn, matplotlib.pyplot


---
## Exercise 1: Data Loading & Provenance (10 points)

Load the dataset from `../sample_data_from_redshift/sample.parquet` and create metadata tracking:

**Tasks**:
1. Load the parquet file
2. Convert date columns (`iddim_date_inicio`, `iddim_date_fim`) to datetime
3. Create a metadata dictionary with:
   - `source`: data source type
   - `file_path`: path to the file
   - `extraction_date`: current timestamp (use `datetime.now(UTC)`)
   - `n_rows`: number of rows
   - `n_columns`: number of columns
   - `date_range`: dict with `min` and `max` dates from `iddim_date_inicio`
4. Print the data shape and metadata

**Hint**: Use `pd.read_parquet()` and `pd.to_datetime()`

In [ ]:
# YOUR CODE HERE
from datetime import datetime, UTC



**Question 1.1**: Why is tracking data provenance important in MLOps?

*YOUR ANSWER HERE*

---
## Exercise 2: Data Overview & Type Conversion (15 points)

**Tasks**:
1. Print the dataset shape
2. Count the number of numeric, categorical, and datetime features BEFORE conversion
3. Convert string columns to numeric where possible (suppress FutureWarning)
4. Count the number of numeric, categorical, and datetime features AFTER conversion
5. Compare before vs after - how many columns changed type?

**Hint**: Use `df.select_dtypes()` with `include='number'`, `include='object'`, `include='datetime'`

In [ ]:
# YOUR CODE HERE


**Question 2.1**: Why might columns be stored as strings when they're actually numeric?

*YOUR ANSWER HERE*

---
## Exercise 3: Data Quality - Completeness (15 points)

**Tasks**:
1. Calculate the number of missing values per column
2. Calculate the percentage of missing values per column
3. Create a DataFrame showing only columns with missing values
4. Sort by missing count (descending)
5. Display the results

**Hint**: Use `df.isna().sum()` and create a DataFrame with both count and percentage

In [ ]:
# YOUR CODE HERE


**Question 3.1**: What threshold would you use to decide if a column has "too many" missing values? Why?

*YOUR ANSWER HERE*

---
## Exercise 4: Data Quality - Duplicates (10 points)

**Tasks**:
1. Check how many duplicate rows exist in the dataset
2. Calculate the percentage of duplicates
3. Print a message with the results
4. If duplicates exist, suggest how to remove them

**Hint**: Use `df.duplicated().sum()`

In [ ]:
# YOUR CODE HERE


---
## Exercise 5: Target Variable Analysis (15 points)

**Tasks**:
1. Calculate the distribution of the target variable (`churn`)
2. Show both counts and percentages
3. Calculate the imbalance ratio (majority class % / minority class %)
4. Determine if the dataset is imbalanced (ratio > 1.5:1)
5. Create a bar plot showing the churn distribution

**Hint**: Use `value_counts()` and plotly express (`px.bar`)

In [ ]:
# YOUR CODE HERE


**Question 5.1**: Why is accuracy NOT a good metric for imbalanced datasets? What metrics should you use instead?

*YOUR ANSWER HERE*

---
## Exercise 6: Univariate Analysis - Summary Statistics (20 points)

**Tasks**:
1. Get all numeric columns (exclude ID columns: those with 'id' or 'codigo' in name)
2. Create summary statistics using `describe()`
3. Add the following columns to the summary:
   - `missing_pct`: percentage of missing values
   - `skewness`: skewness of the distribution
   - `n_zeros`: count of zero values
4. Add a `flags` column that marks:
   - `⚠️HIGH_MISSING` if missing_pct > 20%
   - `📊SKEWED` if abs(skewness) > 2
   - `🔢MANY_ZEROS` if n_zeros > 50% of rows
5. Sort by number of flags (most issues first)
6. Display the top 10 features with the most issues

**Hint**: Use boolean indexing to add flags, e.g., `summary.loc[condition, 'flags'] += 'FLAG '`

In [ ]:
# YOUR CODE HERE


**Question 6.1**: What does high skewness indicate about a feature? How might it affect model training?

*YOUR ANSWER HERE*

---
## Exercise 7: Bivariate Analysis - Feature-Target Correlation (25 points)

This is a more challenging exercise that combines multiple concepts.

**Tasks**:
1. Implement a function `feature_target_correlation(df, target='churn')` that:
   - Excludes ID columns (containing 'id' or 'codigo')
   - Excludes datetime columns
   - Excludes the target column itself
   - For numeric features: calculates point-biserial correlation
   - For categorical features: calculates Cramér's V
   - Returns a DataFrame with columns: `feature`, `correlation`, `method`
2. Use the function to calculate correlations
3. Sort by correlation (descending)
4. Display the top 10 most correlated features
5. Create a simple visualization (your choice) showing the top 3 numeric features vs churn

**Hints**: 
- Import `from scipy.stats import pointbiserialr, chi2_contingency`
- Point-biserial: `corr, _ = pointbiserialr(x, y)`
- Cramér's V formula: `np.sqrt(chi2 / (n * (min(shape) - 1)))`
- Use `abs()` for numeric correlations to get strength regardless of direction

In [ ]:
# YOUR CODE HERE
from scipy.stats import pointbiserialr, chi2_contingency



**Question 7.1**: Why do we use different correlation methods for numeric vs categorical features?

*YOUR ANSWER HERE*

**Question 7.2**: Why do we exclude ID columns from correlation analysis?

*YOUR ANSWER HERE*

---
## Exercise 8: Train/Validation/Test Split (15 points)

**Tasks**:
1. Sort the dataframe by `iddim_date_inicio` (time-based ordering)
2. Split into 80% train, 10% validation, 10% test
3. Print the size of each split (both count and percentage)
4. Print the date range for each split
5. Verify that train dates come before validation dates, and validation before test

**Hint**: Use integer indexing with `.iloc[]`

In [ ]:
# YOUR CODE HERE


**Question 8.1**: Why do we use time-based splitting instead of random splitting? What problem does it prevent?

*YOUR ANSWER HERE*

**Question 8.2**: In production, when would you retrain your model? How would you adjust your train/val/test splits?

*YOUR ANSWER HERE*

---
## Bonus Challenge (10 points)

**Task**: Export your findings as MLOps artifacts

Create a comprehensive report dictionary that includes:
1. Data provenance metadata
2. Data quality summary (% missing, % duplicates)
3. Target distribution and imbalance ratio
4. Top 5 features most correlated with churn
5. Train/val/test split sizes and date ranges

Save this as JSON: `eda_summary_report.json`

**Why**: In production, these artifacts feed into:
- Data validation pipelines
- Model monitoring dashboards
- Experiment tracking systems (MLflow)
- Documentation for compliance

In [ ]:
# YOUR CODE HERE
import json



---
## Reflection Questions

Answer these questions to solidify your understanding:

**1. Data Quality Dimensions**: Name the 5 data quality dimensions we discussed. Give an example of a data quality issue for each dimension from this dataset.

*YOUR ANSWER HERE*

**2. MLOps Level 2**: What does it mean to have "reproducible EDA"? What practices ensure reproducibility?

*YOUR ANSWER HERE*

**3. Pipeline Thinking**: How do the outputs from EDA feed into the next stages of the ML pipeline (training, deployment, monitoring)?

*YOUR ANSWER HERE*

**4. Real-world Application**: Imagine you're presenting EDA findings to:
   - A data scientist: What would you emphasize?
   - A business stakeholder: What would you emphasize?
   - An ML engineer: What would you emphasize?

*YOUR ANSWER HERE*

---
## Submission Checklist

Before submitting, ensure you have:

- [ ] Completed all 8 exercises
- [ ] Answered all discussion questions
- [ ] Run all cells without errors
- [ ] Created at least one visualization
- [ ] Attempted the bonus challenge
- [ ] Answered reflection questions
- [ ] Code follows the patterns from the example notebook
- [ ] Used meaningful variable names
- [ ] Added comments where code is complex

**Grading**: Total 125 points (115 base + 10 bonus)

Good luck! 🚀